<a href="https://colab.research.google.com/github/Aradhyagodambe/quire/blob/main/quire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU llama-index-postprocessor-sbert-rerank pymupdf4llm llama-index transformers accelerate bitsandbytes sentence-transformers llama-index-embeddings-huggingface llama-index-llms-huggingface

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_use_double_quant = True
)

In [ ]:
from transformers import AutoTokenizer

zephyr_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

def messages_to_prompt(messages):
    chat = [{"role": m.role.value, "content": m.content} for m in messages]
    return zephyr_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

def completion_to_prompt(completion):
    return zephyr_tokenizer.apply_chat_template(
        [{"role": "user", "content": completion}], tokenize=False, add_generation_prompt=True
    )

In [ ]:
llm = HuggingFaceLLM(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name="HuggingFaceH4/zephyr-7b-beta",
    context_window=8192,
    max_new_tokens=512,
    model_kwargs={"quantization_config": bnb_config},
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)

In [ ]:
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name = "sentence-transformers/all-MiniLM-L6-v2")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

In [ ]:
import pymupdf4llm
from llama_index.core import Document, VectorStoreIndex
from google.colab import files

In [ ]:
uploaded = files.upload()

documents = []

Saving research ppr 6.pdf to research ppr 6.pdf
Saving research ppr 5.pdf to research ppr 5.pdf
Saving research ppr 4.pdf to research ppr 4.pdf
Saving research ppr 3.pdf to research ppr 3.pdf
Saving research ppr 2.pdf to research ppr 2.pdf
Saving research ppr.pdf to research ppr.pdf


In [ ]:
import re

def clean_filename(pdf_path):

    return re.sub(r" \(\d+\)(?=\.\w+$)", "", pdf_path)

In [ ]:
for pdf_path in uploaded.keys():
  print(pdf_path)

  page_data = pymupdf4llm.to_markdown(pdf_path, page_chunks = True)

  for page in page_data:
    current_page_num = page.get("metadata", {}).get("page",0) +1

    doc = Document(
        text = page["text"],
        metadata = {
            "filename" : clean_filename(pdf_path),
            "page_number" : current_page_num
        }
    )
    documents.append(doc)

research ppr 6.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=11/12.
OCR on page.number=13/14.
research ppr 5.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=5/6.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=13/14.
research ppr 4.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.
OCR on page.number=14/15.
OCR on page.number=18/19.
OCR on page.number=19/20.
 page.number=9/10.
OCR on page.number=10/11.
research ppr 3.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=11/12.
OCR 

In [ ]:
print(f"Building Vector Index {len(documents)} ")
index = VectorStoreIndex.from_documents(documents)

Building Vector Index 103 


In [ ]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

In [ ]:
reranker = SentenceTransformerRerank(
    model = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n = 10
)

shared_memory = ChatMemoryBuffer.from_defaults(token_limit = 1500)

In [ ]:
from llama_index.core import PromptTemplate

# 1. Create a simpler, highly direct prompt for the 7B model
custom_condense_prompt = PromptTemplate(
    "Given the following conversation history and a follow-up question, rewrite the follow-up question into a standalone query that contains all necessary context.\n\n"
    "Chat History:\n"
    "{chat_history}\n\n"
    "Follow Up Input: {question}\n\n"
    "Standalone query:"
)

In [ ]:
def get_chat_engine(target_files=None):

    filters = None

    if target_files:

        if isinstance(target_files, str):
            target_files = [target_files]

        filters = MetadataFilters(
            filters=[
                MetadataFilter(
                    key="filename",
                    value=target_files,
                    operator=FilterOperator.IN
                )
            ]
        )

    return index.as_chat_engine(
        chat_mode="condense_plus_context",
        memory=shared_memory,
        filters=filters,
        similarity_top_k=24,
        node_postprocessors=[reranker],
        condense_prompt = custom_condense_prompt
    )

In [ ]:
global_engine = get_chat_engine(target_files = None)

query_1 = "Summarize the abstracts and conclusions found in these documents."
print(f"\nQuestion :  {query_1}")
response_1 = global_engine.chat(query_1)
print(f"\nAnswer:\n {response_1.response}\n")


targeted_engine = get_chat_engine(target_files="research ppr 2.pdf")

query_2 = "How does this specific paper address those themes?"
print(f"\nQuestion : {query_2}")
response_2 = targeted_engine.chat(query_2)
print(f"\nAnswer:\n {response_2.response}\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Question :  Summarize the abstracts and conclusions found in these documents.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:
 Document 1: "Research PPR 4"

Abstract: The abstract briefly introduces the proposed model for anomaly detection in financial transactions using machine learning techniques. It highlights the use of a confusion matrix to evaluate the model's effectiveness and the importance of reducing overfitting and validation in the context of fraud detection.

Conclusion: The conclusion summarizes the simulation results, which indicate that the proposed model accurately identifies positive and negative classes, with high true positive and true negative rates. The heatmap of the confusion matrix further supports the effectiveness of the model.

Document 2: "Research PPR 2"

Abstract: The abstract explains the features used in the dataset, which include step, type, amount, identification credentials, account balances, and flags for fraud and large transactions. It also mentions the high imbalance in the dataset and the focus on transfer and cash-out transactions.

Conclusion: The conclusion

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:
 The paper "Research PPR 2" addresses the themes of machine learning techniques for fraud detection and data imbalance in the following ways:

1. Dataset: The paper uses a dataset of financial transactions, which includes features such as step, type, amount, identification credentials, account balances, and flags for fraud and large transactions. The dataset is highly imbalanced, with only 0.001% of the transactions being fraudulent.

2. Machine Learning Algorithms: The paper uses five machine learning algorithms, including Bernoulli naïve bayes, multinomial naïve bayes, passive aggressive classifier, stochastic gradient descent, and perceptron, to select the best model for deployment in the live environment. The authors note that the Bayesian algorithm is probabilistic and considers the presumption that a feature under a class does not depend on the other features of that class, considering each feature to be class independent.

3. Experimentation: The paper conducts thorough

In [ ]:
print("\n--- Source Citations for Global Search---")
for i, node in enumerate(response_1.source_nodes, 1):
    filename = node.node.metadata.get("filename", "Unknown")
    page_num = node.node.metadata.get("page_number", "Unknown")
    score = round(node.score, 3) if node.score is not None else "N/A"
    print(f"Source {i}: {filename} (Page {page_num}) - Relevance Score: {score}")


--- Source Citations for Global Search---
Source 1: research ppr 4.pdf (Page 1) - Relevance Score: -6.491
Source 2: research ppr 2.pdf (Page 1) - Relevance Score: -8.347
Source 3: research ppr 3.pdf (Page 1) - Relevance Score: -9.812


In [ ]:
print("\n--- Source Citations for Targeted Search---")
for i, node in enumerate(response_2.source_nodes, 1):
    filename = node.node.metadata.get("filename", "Unknown")
    page_num = node.node.metadata.get("page_number", "Unknown")
    score = round(node.score, 3) if node.score is not None else "N/A"
    print(f"Source {i}: {filename} (Page {page_num}) - Relevance Score: {score}")


--- Source Citations for Targeted Search---
Source 1: research ppr 2.pdf (Page 1) - Relevance Score: -0.55
Source 2: research ppr 2.pdf (Page 1) - Relevance Score: -0.673
Source 3: research ppr 2.pdf (Page 1) - Relevance Score: -0.689


In [ ]:
# import gc
# del llm
# gc.collect()
# torch.cuda.empty_cache()